In [1]:
%pip install pymupdf
%pip install spacy
%pip install sentence-transformers
%pip install pandas tqdm requests
%pip install ollama
# Also ensure Ollama desktop app is running and model is pulled:
# ollama pull qwen2.5:7b-instruct-q4_K_M  (or whichever tag you downloaded)


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


## Ollama + Qwen2.5 7B Q4 — SoilSense RAG

This notebook uses **Ollama** as the LLM backend instead of HuggingFace.
Make sure Ollama is running locally (`ollama serve`) and the model is available:
```
ollama pull qwen2.5:7b-instruct-q4_K_M
```


In [2]:
import torch
import ollama

# Check if GPU is available (used for embeddings only)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Embedding device: {device}")

# Verify Ollama is reachable and model exists
models = ollama.list()
model_names = [m.model for m in models.models]
print(f"Available Ollama models: {model_names}")

OLLAMA_MODEL = "qwen2.5:7b-instruct-q4_K_M"   # ← change tag if yours differs
if not any(OLLAMA_MODEL in m for m in model_names):
    print(f"[WARNING] '{OLLAMA_MODEL}' not found. Run: ollama pull {OLLAMA_MODEL}")
else:
    print(f"[OK] Model '{OLLAMA_MODEL}' is available.")


Embedding device: cuda
Available Ollama models: ['gemma2:9b-instruct-q4_K_M', 'qwen2.5:7b-instruct-q4_K_M']
[OK] Model 'qwen2.5:7b-instruct-q4_K_M' is available.


In [3]:
import os
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# -------------------------------
# PDF URLs
# -------------------------------
NARC_URL = "https://env.narc.gov.np/uploads/documents/1736849201.pdf"
MOALD_URL = (
    "https://giwmscdnone.gov.np/media/pdf_upload/"
    "MOALD-Statical-Book-Magre-2081-Final_wgfs8ph.pdf"
)
FAO_SOIL_URL = (
    "https://openknowledge.fao.org/server/api/core/bitstreams/"
    "1bd0747c-e9d8-4b28-99bf-55684d121e38/content"
)
FAO_GFSAD_URL = (
    "https://openknowledge.fao.org/server/api/core/bitstreams/"
    "587935ca-862d-4471-9496-7f114631225f/content"
)

# -------------------------------
# File names
# -------------------------------
NARC_FILE = "NARC.pdf"
MOALD_FILE = "MOALD.pdf"
FAO_SOIL_FILE = "FAO_SOIL.pdf"
FAO_GFSAD_FILE = "FAO_GFSAD.pdf"

PDF_FILES = [
    (NARC_FILE, NARC_URL),
    (MOALD_FILE, MOALD_URL),
    (FAO_SOIL_FILE, FAO_SOIL_URL),
    (FAO_GFSAD_FILE, FAO_GFSAD_URL),
]


def create_session():
    session = requests.Session()
    retries = Retry(
        total=5,
        backoff_factor=1,
        status_forcelist=[403, 429, 500, 502, 503, 504],
        allowed_methods=["GET"]
    )
    adapter = HTTPAdapter(max_retries=retries)
    session.mount("http://", adapter)
    session.mount("https://", adapter)
    session.headers.update({
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/137.0.0.0 Safari/537.36"
        ),
        "Accept": "application/pdf,application/octet-stream,*/*",
        "Accept-Language": "en-US,en;q=0.9",
        "Referer": "https://www.google.com/",
        "Connection": "keep-alive",
    })
    return session


def download_pdf(file_path, url):
    if os.path.exists(file_path):
        print(f"[INFO] File already exists: {file_path}")
        return
    print(f"[INFO] Downloading: {file_path}")
    session = create_session()
    try:
        response = session.get(url, stream=True, timeout=30, allow_redirects=True)
        response.raise_for_status()
        with open(file_path, "wb") as file:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    file.write(chunk)
        print(f"[SUCCESS] Saved as: {file_path}")
    except requests.exceptions.HTTPError as e:
        print(f"[HTTP ERROR] {file_path}: {e}")
    except requests.exceptions.ConnectionError:
        print(f"[ERROR] Connection failed for {file_path}")
    except requests.exceptions.Timeout:
        print(f"[ERROR] Timeout while downloading {file_path}")
    except Exception as e:
        print(f"[ERROR] Failed to download {file_path}: {e}")


for file_path, url in PDF_FILES:
    download_pdf(file_path, url)

[INFO] File already exists: NARC.pdf
[INFO] File already exists: MOALD.pdf
[INFO] File already exists: FAO_SOIL.pdf
[INFO] File already exists: FAO_GFSAD.pdf


In [4]:
import fitz  # PyMuPDF
from tqdm.auto import tqdm
import random


def text_formatter(text: str) -> str:
    """Performs minor formatting on text."""
    cleaned_text = text.replace("\n", " ").strip()
    return cleaned_text


def open_and_read_pdf(pdf_path: str, source_tag: str = "") -> list:
    """Open a PDF and return a list of page dicts with text and stats."""
    doc = fitz.open(pdf_path)
    pages_and_texts = []
    for page_number, page in tqdm(enumerate(doc), desc=f"Reading {pdf_path}"):
        text = page.get_text()
        text = text_formatter(text=text)
        pages_and_texts.append({
            "source": source_tag,
            "page_number": page_number,
            "page_char_count": len(text),
            "page_word_count": len(text.split(" ")),
            "page_sentence_count_raw": len(text.split(". ")),
            "page_token_count": len(text) / 4,  # token ~ 4 chars
            "text": text
        })
    return pages_and_texts


# Read all 4 PDFs and combine
pages_and_texts = []
for file_path, _ in PDF_FILES:
    source_tag = file_path.replace(".pdf", "")
    pages = open_and_read_pdf(pdf_path=file_path, source_tag=source_tag)
    pages_and_texts.extend(pages)
    print(f"  → {file_path}: {len(pages)} pages")

print(f"\nTotal pages across all PDFs: {len(pages_and_texts)}")
pages_and_texts[:2]

p:\eco\RAG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Reading NARC.pdf: 80it [00:00, 160.71it/s]


  → NARC.pdf: 80 pages


Reading MOALD.pdf: 242it [00:01, 200.98it/s]


  → MOALD.pdf: 242 pages


Reading FAO_SOIL.pdf: 145it [00:00, 281.85it/s]


  → FAO_SOIL.pdf: 145 pages


Reading FAO_GFSAD.pdf: 109it [00:00, 344.26it/s]

  → FAO_GFSAD.pdf: 109 pages

Total pages across all PDFs: 576


[{'source': 'NARC',
  'page_number': 0,
  'page_char_count': 191,
  'page_word_count': 25,
  'page_sentence_count_raw': 1,
  'page_token_count': 47.75,
  'text': '01 Annual Report 2078/79 (2021/22) Government of Nepal  Nepal Agricultural Research Council National Agricultural Environment Research Centre Khumaltar, Lalitpur, Nepal 2022  NPSN: 018/079/80'},
 {'source': 'NARC',
  'page_number': 1,
  'page_char_count': 0,
  'page_word_count': 1,
  'page_sentence_count_raw': 1,
  'page_token_count': 0.0,
  'text': ''}]

In [5]:
import pandas as pd

df = pd.DataFrame(pages_and_texts)
df.head()

,source,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count,text
0,NARC,0,191,25,1,47.75,01 Annual Report 2078/79 (2021/22) Government ...
1,NARC,1,0,1,1,0.00,
2,NARC,2,187,23,1,46.75,Annual Report 2078/79 (2021/22) Government of ...
3,NARC,3,501,61,4,125.25,ii © National Agricultural Environment Researc...
4,NARC,4,2599,397,24,649.75,iii FOREWORD The National Agricultural Environ...


In [6]:
df.describe().round(2)

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count
count,576.00,576.00,576.00,576.00,576.00
mean,84.46,2036.95,396.57,11.10,509.24
std,61.65,996.01,329.76,12.88,249.00
min,0.00,0.00,1.00,1.00,0.00
25%,35.75,1518.00,259.75,2.00,379.50
50%,71.50,2103.00,411.00,6.50,525.75
75%,121.00,2726.00,517.00,18.00,681.50
max,241.00,5870.00,4891.00,142.00,1467.50


In [7]:
from spacy.lang.en import English

nlp = English()
nlp.add_pipe("sentencizer")

# Quick sanity check
doc = nlp("This is a sentence. This is another sentence. I like soil science.")
assert len(list(doc.sents)) == 3
list(doc.sents)

[This is a sentence., This is another sentence., I like soil science.]

In [8]:
for item in tqdm(pages_and_texts, desc="Splitting sentences"):
    item["sentences"] = [str(s) for s in nlp(item["text"]).sents]
    item["page_sentence_count_spacy"] = len(item["sentences"])

# Show a sample
random.sample(pages_and_texts, k=1)

Splitting sentences: 100%|██████| 576/576 [00:04<00:00, 131.17it/s]


[{'source': 'MOALD',
  'page_number': 71,
  'page_char_count': 1572,
  'page_word_count': 257,
  'page_sentence_count_raw': 3,
  'page_token_count': 393.0,
  'text': '63 STATISTICAL INFORMATION ON NEPALESE AGRICULTURE 2079/80 (2022/23) Table 4.4 : Milking Population and its Production by Districts, Fiscal Year 2079/80  (2022/23) DISTRICT BY PROVINCE MILKING  COWS NO.  MILKING  BUFFALOES NO. COW MILK  (Mt) BUFFALOES  MILK (Mt) TOTAL MILK  PRODUCED (Mt) MADHESH PROVINCE 90595 159892 178984 296886 475870 BARA 13059 16941 28006 32145 60151 DHANUSHA 12545 23243 25401 43254 68655 MAHOTTARI 9151 21787 19805 40110 59915 PARSA 6166 8635 13343 16180 29524 RAUTAHAT 7057 19480 15135 36173 51308 SAPTARI 20496 17415 36308 32143 68451 SARLAHI 10563 29766 22655 55382 78037 SIRAHA 11558 22626 18331 41499 59830 BAGMATI PROVINCE 127649 123048 218183 218640 436823 BHAKTAPUR 3843 320 8420 679 9099 CHITWAN 18044 16174 45512 36799 82311 DHADING  10770 15155 20967 22840 43807 DOLAKHA 6234 6725 9389 9774 19163

In [9]:
df = pd.DataFrame(pages_and_texts)
df.describe().round(2)

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count,page_sentence_count_spacy
count,576.00,576.00,576.00,576.00,576.00,576.00
mean,84.46,2036.95,396.57,11.10,509.24,11.12
std,61.65,996.01,329.76,12.88,249.00,12.10
min,0.00,0.00,1.00,1.00,0.00,0.00
25%,35.75,1518.00,259.75,2.00,379.50,2.00
50%,71.50,2103.00,411.00,6.50,525.75,7.00
75%,121.00,2726.00,517.00,18.00,681.50,19.00
max,241.00,5870.00,4891.00,142.00,1467.50,121.00


In [10]:
num_sentence_chunk_size = 10

def split_list(input_list: list, slice_size: int = num_sentence_chunk_size) -> list:
    """Split a list into chunks of slice_size."""
    return [input_list[i:i + slice_size] for i in range(0, len(input_list), slice_size)]

# Quick test
test_list = list(range(25))
split_list(test_list)  # [[0..9], [10..19], [20..24]]

[[0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
 [10, 11, 12, 13, 14, 15, 16, 17, 18, 19],
 [20, 21, 22, 23, 24]]

In [11]:
for item in tqdm(pages_and_texts, desc="Chunking"):
    item["sentence_chunks"] = split_list(input_list=item["sentences"],
                                          slice_size=num_sentence_chunk_size)
    item["num_chunks"] = len(item["sentence_chunks"])

df = pd.DataFrame(pages_and_texts)
df.describe().round(2)

Chunking: 100%|███████████████| 576/576 [00:00<00:00, 67598.96it/s]


,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count,page_sentence_count_spacy,num_chunks
count,576.00,576.00,576.00,576.00,576.00,576.00,576.00
mean,84.46,2036.95,396.57,11.10,509.24,11.12,1.67
std,61.65,996.01,329.76,12.88,249.00,12.10,1.17
min,0.00,0.00,1.00,1.00,0.00,0.00,0.00
25%,35.75,1518.00,259.75,2.00,379.50,2.00,1.00
50%,71.50,2103.00,411.00,6.50,525.75,7.00,1.00
75%,121.00,2726.00,517.00,18.00,681.50,19.00,2.00
max,241.00,5870.00,4891.00,142.00,1467.50,121.00,13.00


In [12]:
import re

pages_and_chunks = []
for item in tqdm(pages_and_texts, desc="Flattening chunks"):
    for sentence_chunk in item["sentence_chunks"]:
        chunk_dict = {}
        chunk_dict["source"] = item["source"]
        chunk_dict["page_number"] = item["page_number"]

        # Join sentences into a paragraph
        joined_sentence_chunk = "".join(sentence_chunk).replace("  ", " ").strip()
        # Add space after period before capital letter
        joined_sentence_chunk = re.sub(r'\.([A-Z])', r'. \1', joined_sentence_chunk)

        chunk_dict["sentence_chunk"] = joined_sentence_chunk
        chunk_dict["chunk_char_count"] = len(joined_sentence_chunk)
        chunk_dict["chunk_word_count"] = len(joined_sentence_chunk.split(" "))
        chunk_dict["chunk_token_count"] = len(joined_sentence_chunk) / 4

        pages_and_chunks.append(chunk_dict)

print(f"Total chunks: {len(pages_and_chunks)}")
random.sample(pages_and_chunks, 1)

Flattening chunks: 100%|███████| 576/576 [00:00<00:00, 9353.37it/s]

Total chunks: 963


[{'source': 'FAO_SOIL',
  'page_number': 83,
  'sentence_chunk': 'A special case is found in depression areas where dryland crops are commonly planted on constructed ridges that alternate with drainage furrows. The original soil profile of the ridge areas is buried under a thick layer of added soil material. The ridge-and-furrow system is known from such different environments as the wet forests of western Europe and the coastal swamps of Southeast Asia where the ridges are planted to dryland crops and rice is grown in the shallow ditch areas. In parts of western Europe, notably in Ireland and the United Kingdom, calcareous materials (e.g. beach sands) were carted to areas with acid Arenosols, Podzols, Albeluvisols and Histosols. Eventually these modified surface layers of mineral material turned into terric horizons that give the soil much improved properties for arable cropping than the original surface soil. In Central Mexico, deep soils were constructed of organic-matter-rich lacus

In [13]:
df = pd.DataFrame(pages_and_chunks)
df.describe().round(2)

,page_number,chunk_char_count,chunk_word_count,chunk_token_count
count,963.00,963.00,963.00,963.00
mean,77.53,1190.14,209.38,297.53
std,53.82,685.25,172.72,171.31
min,0.00,2.00,1.00,0.50
25%,37.50,658.50,101.00,164.62
50%,66.00,1181.00,189.00,295.25
75%,105.00,1644.50,265.50,411.12
max,240.00,3495.00,2516.00,873.75


In [14]:
min_token_length = 30

# Show some short chunks to understand what we're removing
for row in df[df["chunk_token_count"] <= min_token_length].sample(5).iterrows():
    print(f'Tokens: {row[1]["chunk_token_count"]} | Source: {row[1]["source"]} | Text: {row[1]["sentence_chunk"]}')

Tokens: 3.75 | Source: MOALD | Text: MISCELLANEOUS 8
Tokens: 3.0 | Source: MOALD | Text: VEGETABLES 7
Tokens: 20.75 | Source: NARC | Text: सयाउमा लागने भुवादार लाही कीराको व्यवसथापनको लागि नियमीत अनुगमन गरि एकिकृत बयबसथापन
Tokens: 2.5 | Source: FAO_SOIL | Text: 1): 13–19.
Tokens: 0.75 | Source: NARC | Text: xiv


In [15]:
pages_and_chunks_over_min_token_len = df[df["chunk_token_count"] > min_token_length].to_dict(orient="records")
print(f"Chunks remaining after filter: {len(pages_and_chunks_over_min_token_len)}")
pages_and_chunks_over_min_token_len[:2]

Chunks remaining after filter: 927


[{'source': 'NARC',
  'page_number': 0,
  'sentence_chunk': '01 Annual Report 2078/79 (2021/22) Government of Nepal Nepal Agricultural Research Council National Agricultural Environment Research Centre Khumaltar, Lalitpur, Nepal 2022 NPSN: 018/079/80',
  'chunk_char_count': 189,
  'chunk_word_count': 23,
  'chunk_token_count': 47.25},
 {'source': 'NARC',
  'page_number': 2,
  'sentence_chunk': 'Annual Report 2078/79 (2021/22) Government of Nepal Nepal Agricultural Research Council National Agricultural Environment Research Centre Khumaltar, Lalitpur, Nepal 2022 NPSN: 018/079/80',
  'chunk_char_count': 186,
  'chunk_word_count': 22,
  'chunk_token_count': 46.5}]

In [16]:
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

embedding_model = SentenceTransformer(
    model_name_or_path="all-mpnet-base-v2",
    device=device
)

# Quick test
test_embed = embedding_model.encode(["soil pH affects crop yield"])
print(f"Embedding shape: {test_embed.shape}")

Using device: cuda


Loading weights: 100%|█████████| 199/199 [00:00<00:00, 4263.35it/s]


Embedding shape: (1, 768)


In [17]:
# Embed all chunks (batched for speed)
text_chunks = [item["sentence_chunk"] for item in pages_and_chunks_over_min_token_len]
print(f"Embedding {len(text_chunks)} chunks...")

text_chunk_embeddings = embedding_model.encode(
    text_chunks,
    batch_size=16,
    convert_to_tensor=True,
    show_progress_bar=True
)
print(f"Embeddings shape: {text_chunk_embeddings.shape}")

Embedding 927 chunks...


Batches: 100%|█████████████████████| 58/58 [00:15<00:00,  3.71it/s]

Embeddings shape: torch.Size([927, 768])


In [18]:
# Also store embedding per chunk item (for CSV save)
for i, item in enumerate(pages_and_chunks_over_min_token_len):
    item["embedding"] = text_chunk_embeddings[i].cpu().numpy()

In [19]:
text_chunks_and_embeddings_df = pd.DataFrame(pages_and_chunks_over_min_token_len)
embeddings_df_save_path = "text_chunks_and_embeddings_df.csv"
text_chunks_and_embeddings_df.to_csv(embeddings_df_save_path, index=False)
print(f"Saved {len(text_chunks_and_embeddings_df)} rows to {embeddings_df_save_path}")
text_chunks_and_embeddings_df.head()

Saved 927 rows to text_chunks_and_embeddings_df.csv


,source,page_number,sentence_chunk,chunk_char_count,chunk_word_count,chunk_token_count,embedding
0,NARC,0,01 Annual Report 2078/79 (2021/22) Government ...,189,23,47.25,"[0.051609494, -0.010961796, -0.017161343, 0.02..."
1,NARC,2,Annual Report 2078/79 (2021/22) Government of ...,186,22,46.50,"[0.05293816, -0.014074279, -0.014764885, 0.019..."
2,NARC,3,ii © National Agricultural Environment Researc...,496,56,124.00,"[0.06096496, -0.019012248, -0.021903904, 0.024..."
3,NARC,4,iii FOREWORD The National Agricultural Environ...,1628,233,407.00,"[0.025994955, -0.016698133, -0.002780142, 0.03..."
4,NARC,4,"Researchers, extension staff, students, and na...",940,134,235.00,"[0.045338605, -0.022865236, -0.01672553, 0.025..."


In [20]:
import numpy as np
import torch
import pandas as pd

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load from CSV
text_chunks_and_embeddings_df = pd.read_csv("text_chunks_and_embeddings_df.csv")

# Convert string representation back to numpy array
text_chunks_and_embeddings_df["embedding"] = text_chunks_and_embeddings_df["embedding"].apply(
    lambda x: np.fromstring(x.strip("[]"), sep=" ")
)

# Stack into a torch tensor on GPU/CPU
embeddings = torch.tensor(
    np.stack(text_chunks_and_embeddings_df["embedding"].tolist(), axis=0),
    dtype=torch.float32
).to(device)

# Convert to list of dicts for lookup
pages_and_chunks = text_chunks_and_embeddings_df.to_dict(orient="records")

print(f"Embeddings shape: {embeddings.shape}")
print(f"Total chunks loaded: {len(pages_and_chunks)}")

Embeddings shape: torch.Size([927, 768])
Total chunks loaded: 927


In [21]:
# Reload embedding model
from sentence_transformers import SentenceTransformer, util

embedding_model = SentenceTransformer(
    model_name_or_path="all-mpnet-base-v2",
    device=device
)

Loading weights: 100%|█████████| 199/199 [00:00<00:00, 7624.47it/s]


In [22]:
from time import perf_counter as timer
import textwrap

def print_wrap(text: str, width: int = 100):
    wrapped = textwrap.fill(text, width=width)
    print(wrapped)


def retrieve_relevant_resources(
    query: str,
    embeddings: torch.Tensor,
    model: SentenceTransformer = embedding_model,
    n_resources_to_return: int = 5,
    print_time: bool = True
):
    """
    Embed a query and return the top-k scores and indices from the embedding store.
    """
    query_embedding = model.encode(query, convert_to_tensor=True)

    start_time = timer()
    dot_scores = util.dot_score(query_embedding, embeddings)[0]
    end_time = timer()

    if print_time:
        print(f"[INFO] Time taken to get scores on ({len(embeddings)} embeddings : {end_time - start_time:.5f} seconds.)")

    scores, indices = torch.topk(input=dot_scores, k=n_resources_to_return)
    return scores, indices


def print_top_results_and_scores(
    query: str,
    embeddings: torch.Tensor,
    pages_and_chunks: list = pages_and_chunks,
    n_resources_to_return: int = 5
):
    scores, indices = retrieve_relevant_resources(
        query=query, embeddings=embeddings, n_resources_to_return=n_resources_to_return
    )
    for score, idx in zip(scores, indices):
        print(f"Score: {score:.4f} | Source: {pages_and_chunks[idx]['source']} | Page: {pages_and_chunks[idx]['page_number']}")
        print("Text:")
        print_wrap(pages_and_chunks[idx]["sentence_chunk"])
        print("\n")

In [23]:
# Test retrieval
query = "What is the best crop for clay loam soil in Nepal?"
print(f"Query: {query}")
print_top_results_and_scores(query=query, embeddings=embeddings)

Query: What is the best crop for clay loam soil in Nepal?
[INFO] Time taken to get scores on (927 embeddings : 0.00661 seconds.)
Score: 0.6115 | Source: MOALD | Page: 34
Text:
26 STATISTICAL INFORMATION ON NEPALESE AGRICULTURE 2079/80 (2022/23) TABLE 2.3: Cash Crops by
Districts, Fiscal Year 2079/80 (2022/23) Area in Hectare(Ha), Production in Metric Tonnes(Mt), and
Yield in Mt/Ha Province  District Oilseed Sugarcane Potato Area  Prod. Yield Area  Prod. Yield  Area
Prod. Yield LUMBINI KAPILBASTU 4,270  4,255 1.00 3,150  150,800 47.87  2,592  38,920 15.02 LUMBINI
DANG 16,436 18,443 1.12  28 896 32.00  2,535  40,365 15.92 LUMBINI BANKE 7,330  7,107 0.97  42 1,750
41.67  3,210  43,335 13.50 LUMBINI BARDIYA 15,030 17,251 1.15 210 9,870 47.00  4,300  70,950 16.50
LUMBINI RUKUM EAST 383  371 0.97  3 142 47.33  1,204  22,595 18.77 LUMBINI PYUTHAN 990  934 0.94  11
520 47.27  1,660  26,052 15.69 LUMBINI ROLPA 500  491 0.98  3 70 28.00  2,075  34,653 16.70  
LUMBINI 60,567 64,453 1.06 7,361  34

In [24]:
# Another test
query = "fertilizer urea application Nepal"
print(f"Query: {query}")
print_top_results_and_scores(query=query, embeddings=embeddings)

Query: fertilizer urea application Nepal
[INFO] Time taken to get scores on (927 embeddings : 0.00023 seconds.)
Score: 0.6695 | Source: MOALD | Page: 184
Text:
176 STATISTICAL INFORMATION ON NEPALESE AGRICULTURE 2079/80 (2022/23) Table 9.1: ANNUAL SALES OF
CHEMICAL FERTILIZER, 2010/11 - 2022/23 Unit: Mt. Fertilizer: Type 2010/11 2011/12 2012/13 2013/14
2014/15 2015/16 2016/17 2017/18 2018/19 2019/20 2020/21 2021/22 2022/23 Urea 85,191 97,957  108,553
145,622  190,163  213,063  205,425  235,304  215,733  224,700  225,180  143,482  226,148 DAP 22,001
43,146 65,722 81,520  101,797  107,121  114,802  105,619  120,893  160,298  140,982 77,720  110,120
Potash 2,821  3,711  2,688  5,046  6,716  7,336  7,991  7,811  7,377  9,597  12,990  6,633  6,455
Total AICL& STCL 110,013  144,813  176,963  232,189  298,677  327,520  328,217  348,734  344,004
394,595  379,152  227,836  342,723


Score: 0.6514 | Source: MOALD | Page: 185
Text:
177 STATISTICAL INFORMATION ON NEPALESE AGRICULTURE 2079/80 (2022

In [25]:
def dot_product(vector1, vector2):
    return torch.dot(vector1, vector2)

def cosine_similarity(vector1, vector2):
    dot = torch.dot(vector1, vector2)
    norm1 = torch.sqrt(torch.sum(vector1 ** 2))
    norm2 = torch.sqrt(torch.sum(vector2 ** 2))
    return dot / (norm1 * norm2)

v1 = torch.tensor([1, 2, 3], dtype=torch.float32)
v2 = torch.tensor([1, 2, 3], dtype=torch.float32)
v3 = torch.tensor([4, 5, 6], dtype=torch.float32)

print("Dot product v1·v2:", dot_product(v1, v2))
print("Dot product v1·v3:", dot_product(v1, v3))
print("Cosine sim v1,v2:", cosine_similarity(v1, v2))
print("Cosine sim v1,v3:", cosine_similarity(v1, v3))

Dot product v1·v2: tensor(14.)
Dot product v1·v3: tensor(32.)
Cosine sim v1,v2: tensor(1.0000)
Cosine sim v1,v3: tensor(0.9746)


## LLM Setup — Ollama

We skip GPU-memory-based model selection entirely.
Ollama manages memory, quantization (Q4), and inference internally.


In [26]:
# Model is already set above; just confirm
print(f"LLM backend : Ollama")
print(f"Model       : {OLLAMA_MODEL}")
print(f"Quantization: Q4 (managed by Ollama)")


LLM backend : Ollama
Model       : qwen2.5:7b-instruct-q4_K_M
Quantization: Q4 (managed by Ollama)


In [50]:
import ollama

def ollama_generate(prompt: str, temperature: float = 0.7, max_new_tokens: int = 512) -> str:
    response = ollama.chat(
        model=OLLAMA_MODEL,
        messages=[{"role": "user", "content": prompt}],
        options={
            "temperature": temperature,
            "num_predict": max_new_tokens,
            "num_ctx": 4096,
            "stop": ["\n\n\n", "Note:", "Remember:"]  # ← cuts rambling endings
        }
    )
    return response["message"]["content"]

# Quick smoke test
test_reply = ollama_generate("Say 'Ollama is ready' in one sentence.")
print(test_reply)


Ollama is ready.


*(Model parameter/memory stats are not applicable when using Ollama — the model runs as a separate process.)*


In [28]:
# Confirm embedding device
print(f"Embedding device: {device}")


Embedding device: cuda


In [29]:
input_text = "What are the main cereal crops grown in Nepal?"
print(f"Input text: {input_text}")

# With Ollama we pass the message directly — no manual chat-template needed
reply = ollama_generate(input_text)
print(f"\nResponse:\n{reply}")


Input text: What are the main cereal crops grown in Nepal?

Response:
Nepal's agricultural sector is dominated by cereal crops due to its diverse agro-ecological zones and favorable climate for cultivation of various types of grain. The main cereal crops grown in Nepal include:

1. **Rice**: Rice is a major staple food crop, especially in the lowland Terai region where it thrives in the warm and humid conditions.

2. **Wheat**: Wheat is widely cultivated throughout the country, with significant production in the Hill and Mountain regions. It is an important source of dietary fiber and protein.

3. **Barley**: Barley is grown at higher altitudes, particularly in the mountainous regions where it can tolerate colder temperatures.

4. **Maize (Corn)**: Maize is another staple crop that is cultivated across various elevations but predominantly in the middle hills and lowlands.

5. **Millets** (e.g., Foxtail millet, Proso millet): Millets are drought-resistant crops that grow well in arid co

In [30]:
from time import perf_counter as timer

start = timer()
reply = ollama_generate("What are the main cereal crops grown in Nepal?", max_new_tokens=256)
end = timer()
print(f"Response time: {end - start:.2f}s")
print(reply)


Response time: 16.83s
Nepal has diverse agricultural practices due to its varied geographical and climatic conditions. The main cereal crops grown in different regions of Nepal include:

1. **Rice**: Rice is widely cultivated across various altitudes, but it's particularly important in the Terai region, which has a subtropical climate suitable for rice cultivation.

2. **Wheat**: Wheat is a major crop in high-altitude areas such as the hills and mountains, where the cooler temperatures are more favorable for its growth compared to lowland regions.

3. **Maize (Corn)**: Maize is widely grown across different altitudes but is particularly important in the mid-hills and lower mountainous regions of Nepal.

4. **Barley**: Barley is a staple crop in the mountain region where other crops struggle due to harsher climatic conditions.

5. **Millet**: Millet, including varieties like foxtail millet (Setaria italica) and proso millet (Panicum miliaceum), are grown in various parts of Nepal, espec

In [41]:
from dataclasses import dataclass
from typing import Optional

@dataclass
class SoilSenseInput:
    # From ResNet
    soil_type: str           # e.g. "Red_Soil"
    confidence: float        # e.g. 0.78

    # From GPS / device
    location: str            # e.g. "Kathmandu Valley, Nepal"
    elevation_m: float       # e.g. 1340

    # From weather API
    temp_celsius: float
    humidity_percent: float
    rainfall_last_7d_mm: float
    rainfall_forecast_14d_mm: float
    season: str              # e.g. "Pre-monsoon transitional"
    days_since_last_rain: int

    # From user
    query: str

In [51]:
def prompt_formatter(input_data: SoilSenseInput, context_items: list) -> str:

    soil_props = SOIL_PROPERTIES.get(input_data.soil_type, {})
    soil_name = input_data.soil_type.replace("_", " ")

    soil_block = ""
    if soil_props:
        soil_block = (
            f"Soil: {soil_name} | pH: {soil_props['pH_range']} | "
            f"N: {soil_props['Nitrogen_N']} | P: {soil_props['Phosphorus_P']} | "
            f"K: {soil_props['Potassium_K']} | Confidence: {input_data.confidence*100:.0f}%"
        )

    weather_block = (
        f"Location: {input_data.location} | Elevation: {input_data.elevation_m}m | "
        f"Season: {input_data.season} | Temp: {input_data.temp_celsius}°C | "
        f"Humidity: {input_data.humidity_percent}% | "
        f"Rain last 7d: {input_data.rainfall_last_7d_mm}mm | "
        f"Forecast 14d: {input_data.rainfall_forecast_14d_mm}mm | "
        f"Days since rain: {input_data.days_since_last_rain}"
    )

    # Take only top 3 chunks instead of 5, and truncate each to 200 chars
    top_chunks = context_items[:3]
    pdf_context = "\n".join([
        f"- {item['sentence_chunk'][:200]}..." for item in top_chunks
    ])

    confidence_note = (
        f" (low confidence: {input_data.confidence*100:.0f}%, treat soil type as approximate)"
        if input_data.confidence < 0.80 else ""
    )

    prompt = f"""You are an agricultural advisor for Nepal. Answer concisely in 3-5 bullet points max.
No explanations unless critical. Be specific with quantities and timing.
{confidence_note}

CONTEXT:
{soil_block}
{weather_block}

DOCUMENTS:
{pdf_context}

QUERY: {input_data.query}

ANSWER (bullet points only, max 5):"""

    return prompt

In [44]:
def ask(
    input_data: SoilSenseInput,
    temperature: float = 0.7,
    max_new_tokens: int = 512,      # increased — richer context = longer answers
    return_answer_only: bool = True
):
    ## RETRIEVAL — use query + soil type for better hits
    enriched_query = f"{input_data.query} {input_data.soil_type.replace('_', ' ')} Nepal"
    scores, indices = retrieve_relevant_resources(query=enriched_query, embeddings=embeddings)
    context_items = [pages_and_chunks[i] for i in indices]
    for i, item in enumerate(context_items):
        item["score"] = scores[i].cpu()

    ## AUGMENT
    prompt = prompt_formatter(input_data=input_data, context_items=context_items)

    ## GENERATE
    answer = ollama_generate(prompt, temperature=temperature, max_new_tokens=max_new_tokens)

    return answer if return_answer_only else (answer, context_items)


In [48]:
import json

with open("soil_prop.json") as f:
    SOIL_PROPERTIES = json.load(f)

In [52]:
# ── Test 1: Crop recommendation with full context ─────────────────────────
input1 = SoilSenseInput(
    soil_type="Red_Soil", confidence=0.78,
    location="Kathmandu Valley, Nepal", elevation_m=1340,
    temp_celsius=24, humidity_percent=68,
    rainfall_last_7d_mm=12, rainfall_forecast_14d_mm=45,
    season="Pre-monsoon transitional", days_since_last_rain=2,
    query="What crops should I grow and what soil treatments are needed?"
)
print(f"Query: {input1.query}")
print_wrap(ask(input_data=input1, temperature=0.4))

Query: What crops should I grow and what soil treatments are needed?
[INFO] Time taken to get scores on (927 embeddings : 0.00024 seconds.)
- **Crops**: Grow wheat or barley. These crops thrive in the given soil conditions with moderate
potassium levels. - **Soil Treatment**: Apply 100 kg/ha of diammonium phosphate (DAP) to improve
phosphorus levels. - **Irrigation**: Consider irrigation as recent rainfall is low, and forecast
indicates more rain in 2 weeks. - **Fertilizer Application**: Add 50 kg/ha of urea for nitrogen,
given the low nitrogen content. - **Seeding Time**: Plant seeds within the next week to capitalize
on upcoming rains.
